# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Secret successfully loaded!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    'dim_clients':f"read_parquet('{rel}/dim_clients.parquet')",
    'dim_content':f"read_parquet('{rel}/dim_content.parquet')",
    'fact_content_daily_performance':f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    'fact_content_query_90d':f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}


Secret successfully loaded!


In [2]:
df = con.sql(f"""
WITH content_id_hv_criteria AS (
    SELECT
        content_hash_id
    FROM {TABLES['fact_content_daily_performance']}
    WHERE
        gsc_data_available IS TRUE
        AND ga4_data_available IS NOT NULL
    GROUP BY content_hash_id
    HAVING
        COUNT(DISTINCT report_date) >= 120
        AND
        DATE_DIFF(
            'day',
            MIN(report_date),
            MAX(report_date)
        ) + 1 = COUNT(DISTINCT report_date)
)

SELECT fact.*
FROM {TABLES['fact_content_daily_performance']} AS fact
INNER JOIN content_id_hv_criteria AS criteria
    ON fact.content_hash_id = criteria.content_hash_id
WHERE
    fact.gsc_data_available IS TRUE
    AND fact.ga4_data_available IS NOT NULL;
""").df()

df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-02-10,client_9958f0a7ae1df715,content_92fd77e367253da2,True,True,True,False,4,0,29,...,0,0,0,0,0,0,0,0,0,2025-02
1,2025-02-10,client_9958f0a7ae1df715,content_00dafe96928080fb,True,True,True,False,18,0,88,...,0,0,0,0,0,0,0,0,0,2025-02
2,2025-02-10,client_9958f0a7ae1df715,content_7a1bdd56e533fef5,True,True,True,False,14,0,122,...,0,0,0,0,0,0,0,0,0,2025-02
3,2025-02-10,client_9958f0a7ae1df715,content_d73205f6d972a269,True,True,True,False,33,3,127,...,0,0,0,0,0,0,0,0,0,2025-02
4,2025-02-10,client_9958f0a7ae1df715,content_c936b727ae06f104,True,True,True,False,18,0,258,...,0,0,0,0,0,0,0,0,0,2025-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2562564,2026-06-30,client_65de48885f4ef01b,content_f862eedcac7d187e,True,True,True,False,25,0,201,...,0,0,0,0,0,0,0,0,0,2026-06
2562565,2026-06-30,client_65de48885f4ef01b,content_153da632dd6adb12,True,True,True,False,44,0,355,...,0,0,0,0,0,0,0,0,0,2026-06
2562566,2026-06-30,client_65de48885f4ef01b,content_4e0c7263aa6aab62,True,True,True,False,63,1,350,...,0,0,0,0,0,0,0,0,0,2026-06
2562567,2026-06-30,client_65de48885f4ef01b,content_fa9131f1743ae0ae,True,True,True,False,73,0,376,...,0,0,0,0,0,0,0,0,0,2026-06


In [149]:
result = df.groupby('content_hash_id').agg(start_date = ('report_date', "min"),end_date = ('report_date', "max")).sort_values(['start_date','end_date'], ascending=[False,True])
result

,start_date,end_date
content_hash_id,,
content_0060ca245492494f,2026-03-03,2026-06-30
content_00823406ee06d1e7,2026-03-03,2026-06-30
content_0084ee8efa2fcd23,2026-03-03,2026-06-30
content_00e0da358d5df2e7,2026-03-03,2026-06-30
content_00efcc9c1c9bba3d,2026-03-03,2026-06-30
...,...,...
content_9987be00b8c14a02,2025-01-27,2026-06-30
content_cf0500bbb61cc8b9,2025-01-27,2026-06-30
content_d02be57d816cf3d7,2025-01-27,2026-06-30


In [150]:
import pandas as pd

In [151]:
incomplete_content_month = result.copy().reset_index()

incomplete_content_month['incomplete_start_month'] = (
    incomplete_content_month['start_date'].dt.day > 15
)

incomplete_content_month['incomplete_end_month'] = (
    incomplete_content_month['end_date'].dt.day < 15
)

incomplete_content_month['start_month'] = (
    incomplete_content_month['start_date'].dt.to_period('M')
)

incomplete_content_month['end_month'] = (
    incomplete_content_month['end_date'].dt.to_period('M')
)

df['month'] = df['report_date'].dt.to_period('M')

data_complete_month = df.merge(
    incomplete_content_month[
        [
            'content_hash_id',
            'start_month',
            'end_month',
            'incomplete_start_month',
            'incomplete_end_month'
        ]
    ],
    on='content_hash_id',
    how='left'
)

data_complete_month = data_complete_month[
    ~(
        (
            data_complete_month['incomplete_start_month'] &
            (data_complete_month['month'] == data_complete_month['start_month'])
        ) |
        (
            data_complete_month['incomplete_end_month'] &
            (data_complete_month['month'] == data_complete_month['end_month'])
        )
    )
].copy()

In [152]:
data_complete_month.groupby('content_hash_id').agg(days = ('report_date', "count")).sort_values('days',ascending=True)

,days
content_hash_id,
content_eb25f969a5734bc1,109
content_9d0a8af767ca01f3,110
content_93cc7359119173d8,110
content_27283b2c07a61c28,110
content_8f3e71fbfe536666,110
...,...
content_1c7ae2a4d8295008,515
content_f3a75d8cf58dd50b,515
content_84f5a9ecfefa108e,515


In [153]:
data_complete_month[data_complete_month['content_hash_id'] == 'content_f3ba8c1f94eb36f5']

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,start_month,end_month,incomplete_start_month,incomplete_end_month
6213,2025-05-25,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,15,0,330,...,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
6867,2025-05-15,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,22,0,395,...,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
7060,2025-05-20,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,18,0,214,...,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
7603,2025-05-12,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,7,1,28,...,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
7806,2025-05-22,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,19,0,240,...,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2492334,2026-06-13,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,9,0,167,...,0,0,0,0,0,2026-06,2025-05,2026-06,False,False
2492722,2026-06-20,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,2,0,40,...,0,0,0,0,0,2026-06,2025-05,2026-06,False,False
2537164,2026-06-26,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,1,0,34,...,0,0,0,0,0,2026-06,2025-05,2026-06,False,False
2546153,2026-06-23,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,True,True,True,False,7,0,131,...,0,0,0,0,0,2026-06,2025-05,2026-06,False,False


In [154]:
data_complete_month.drop(['start_month', 'end_month','incomplete_start_month','incomplete_end_month'], axis=1, inplace=True)

In [155]:
grouped = data_complete_month.groupby([ 'client_hash_id','content_hash_id', 'month'])

numeric_cols = data_complete_month.select_dtypes(include='number').columns.drop('gsc_avg_position')
bool_cols = data_complete_month.select_dtypes(include='bool').columns

content_month = grouped.agg({
    **{col: 'sum' for col in numeric_cols},
    **{col: 'any' for col in bool_cols},
    'gsc_avg_position':'mean'
}).reset_index()

rename_dict = {
    col: f"{col}_30d" for col in numeric_cols.append(pd.Index(['gsc_avg_position']))
}
content_month = content_month.rename(columns=rename_dict)

In [156]:
content_month

,client_hash_id,content_hash_id,month,gsc_impressions_30d,gsc_clicks_30d,gsc_sum_position_30d,ga4_pageviews_30d,ga4_sessions_30d,ga4_users_30d,ga4_engaged_sessions_30d,...,ai_copilot_30d,ai_claude_30d,ai_meta_30d,ai_other_30d,scroll_events_30d,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_avg_position_30d
0,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-03,330,1,2543,0,0,0,0,...,0,0,0,0,0,True,True,True,False,7.675690
1,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-04,393,5,2817,5,5,4,0,...,0,0,0,0,0,True,True,True,True,7.441862
2,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-05,872,1,6082,1,1,1,0,...,0,0,0,0,0,True,True,True,True,7.100545
3,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-06,1437,6,8742,8,8,8,1,...,0,0,0,0,1,True,True,True,True,6.115233
4,client_0fa64a184f18a4a0,content_003959d7e5063269,2026-03,464,1,4756,1,1,1,0,...,0,0,0,0,0,True,True,True,True,8.768573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85139,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-02,939,4,8060,5,5,5,0,...,0,0,0,0,0,True,True,True,True,8.860769
85140,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-03,1534,3,13546,6,6,6,0,...,0,0,0,0,0,True,True,True,True,9.573120
85141,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-04,482,0,9770,4,4,3,0,...,0,0,0,0,0,True,True,True,True,20.424837
85142,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-05,269,0,8335,5,5,5,1,...,0,0,0,0,1,True,True,True,True,32.566968


In [157]:
cols = content_month.columns.drop(['client_hash_id', 'content_hash_id', 'month', 'client_has_gsc','client_has_ga4', 'gsc_data_available', 'ga4_data_available'])
cols

Index(['gsc_impressions_30d', 'gsc_clicks_30d', 'gsc_sum_position_30d',
       'ga4_pageviews_30d', 'ga4_sessions_30d', 'ga4_users_30d',
       'ga4_engaged_sessions_30d', 'ga4_total_engagement_sec_30d',
       'sessions_organic_30d', 'sessions_direct_30d', 'sessions_referral_30d',
       'sessions_social_30d', 'sessions_paid_30d', 'sessions_ai_30d',
       'ai_chatgpt_30d', 'ai_perplexity_30d', 'ai_gemini_30d',
       'ai_copilot_30d', 'ai_claude_30d', 'ai_meta_30d', 'ai_other_30d',
       'scroll_events_30d', 'gsc_avg_position_30d'],
      dtype='object')

In [158]:
content_month = content_month.sort_values(
    ['client_hash_id', 'content_hash_id', 'month']
).copy()

grouped = content_month.groupby(
    ['client_hash_id', 'content_hash_id']
)

for col in cols:
    base_col = col[:-4]

    content_month[f'{base_col}_prev_30d'] = grouped[col].shift(1)
    content_month[f'{base_col}_2m_ago_30d'] = grouped[col].shift(2)
    content_month[f'{base_col}_future_30d'] = grouped[col].shift(-1)

In [193]:
content_month[content_month['content_hash_id'] == 'content_f3baf9504688']

,client_hash_id,content_hash_id,month,gsc_impressions_30d,gsc_clicks_30d,gsc_sum_position_30d,ga4_pageviews_30d,ga4_sessions_30d,ga4_users_30d,ga4_engaged_sessions_30d,...,ai_meta_future_30d,ai_other_prev_30d,ai_other_2m_ago_30d,ai_other_future_30d,scroll_events_prev_30d,scroll_events_2m_ago_30d,scroll_events_future_30d,gsc_avg_position_prev_30d,gsc_avg_position_2m_ago_30d,gsc_avg_position_future_30d


In [161]:
dataset = content_month[
    content_month['gsc_impressions_2m_ago_30d'].notna() &
    content_month['gsc_impressions_future_30d'].notna()
].copy()

In [168]:
contents = con.sql(f"""
SELECT *
FROM {TABLES['dim_content']}
""").df()

contents

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519601,client_fef1a8f436438636,content_9643d3c771bd23bd,keyword_607ded80eddd6ba8,url_7dd261ac267fd25e,33,7,113,2026-03-06,2026-06-17,keyword article,...,3,2026-03-04,openai,gpt-4o-mini,9694,1492,NaT,NaT,True,False
519602,client_fef1a8f436438636,content_9646b5037272bf2b,keyword_7a2a8d1621719cc4,url_80330f0c93e0093f,51,10,94,2025-12-17,2026-06-17,keyword article,...,0,2025-10-29,None,gpt-4o-mini,9005,1403,NaT,NaT,True,False
519603,client_fef1a8f436438636,content_964d01d55c1d3beb,keyword_1938948b2d97a11d,url_e03c4e008928c4fe,50,10,93,2025-08-03,2026-06-17,keyword article,...,0,2025-08-03,None,gpt-4o-mini,9359,1491,NaT,NaT,True,False
519604,client_fef1a8f436438636,content_965127aa4bcd558d,keyword_11961befb290ac78,url_c708352fc793c780,33,8,129,2025-08-03,2026-06-17,keyword article,...,0,2025-08-03,None,gpt-4o-mini,8737,1446,NaT,NaT,True,False


In [173]:
contents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 519606 entries, 0 to 519605
Data columns (total 26 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   client_hash_id              519606 non-null  object        
 1   content_hash_id             519606 non-null  object        
 2   keyword_hash_id             447608 non-null  object        
 3   url_hash_id                 513081 non-null  object        
 4   keyword_char_count          519606 non-null  int64         
 5   keyword_token_count         519606 non-null  int64         
 6   url_char_count              519606 non-null  int64         
 7   content_created_date        519606 non-null  datetime64[us]
 8   content_updated_date        519606 non-null  datetime64[us]
 9   content_type                519606 non-null  object        
 10  search_volume               376984 non-null  Int64         
 11  competition                 376984 non-

In [169]:
dataset = dataset.merge(contents, on='content_hash_id', how='inner')

In [184]:
dataset.isna().sum().sort_values(ascending=False)['last_optimized_date']

np.int64(16917)

In [172]:
contents

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519601,client_fef1a8f436438636,content_9643d3c771bd23bd,keyword_607ded80eddd6ba8,url_7dd261ac267fd25e,33,7,113,2026-03-06,2026-06-17,keyword article,...,3,2026-03-04,openai,gpt-4o-mini,9694,1492,NaT,NaT,True,False
519602,client_fef1a8f436438636,content_9646b5037272bf2b,keyword_7a2a8d1621719cc4,url_80330f0c93e0093f,51,10,94,2025-12-17,2026-06-17,keyword article,...,0,2025-10-29,None,gpt-4o-mini,9005,1403,NaT,NaT,True,False
519603,client_fef1a8f436438636,content_964d01d55c1d3beb,keyword_1938948b2d97a11d,url_e03c4e008928c4fe,50,10,93,2025-08-03,2026-06-17,keyword article,...,0,2025-08-03,None,gpt-4o-mini,9359,1491,NaT,NaT,True,False
519604,client_fef1a8f436438636,content_965127aa4bcd558d,keyword_11961befb290ac78,url_c708352fc793c780,33,8,129,2025-08-03,2026-06-17,keyword article,...,0,2025-08-03,None,gpt-4o-mini,8737,1446,NaT,NaT,True,False


In [197]:
dataset['optimization_eligible_date'].isna()

,optimization_eligible_date
0,True
1,True
2,True
3,True
4,True
...,...
45014,True
45015,True
45016,True
45017,True


In [165]:
dataset.to_csv("dataset_flyrank")

In [ ]:
feature_cols = [
    'gsc_impressions', //
    'gsc_clicks', //
    'gsc_avg_position',//

    'sessions_organic',//

    'search_volume', //
    'competition',//
    'cpc',//
    'main_intent',//
    'competition_level',//
    'content_type',//
    'word_count',//
    'char_count',//
    'category_count',//
    'backlinks',//

    'content_age_days', s
    'days_since_updated', s
    'days_since_optimized', s
    'has_optimization_date', s

    'ga4_engaged_sessions',
    'scroll_events',
    'page_views'
]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.